# Lasso Regression — L1 Regularization

## 1. Definition and intuition
Lasso regression minimizes squared prediction error plus an L1 penalty on coefficients. Unlike Ridge, it can set some coefficients exactly to zero, producing a sparse model.

## 2. Objective
\\[\\min_{w,b}\\; \\frac{1}{2n}\\|y-Xw-b\\mathbf{1}\\|_2^2 + \\alpha\\|w\\|_1.\\]

## 3. Why it works: derivation and assumptions
The L1 norm is not differentiable at zero, so coordinate descent uses the soft-thresholding operator \\(S(z,\\lambda)=\\mathrm{sign}(z)\\max(|z|-\\lambda,0)\\). Holding other coefficients fixed yields \\(w_j\\leftarrow S(x_j^Tr_{-j},n\\alpha)/(x_j^Tx_j)\\), where \\(r_{-j}\\) is the residual with the current contribution of feature j removed. The objective is convex, so coordinate descent can converge to a global minimizer, though correlated features may make the selected solution unstable.

**Assumptions and failure modes:** Lasso is useful when a sparse linear explanation is plausible. It can arbitrarily select among highly correlated predictors and may underperform when many small effects are real. Feature scaling is important.

## 4. Dataset and preprocessing

We use a synthetic regression task with informative features and correlated columns. Predictors are standardized using statistics fit on the training split only. This prevents test-set leakage and makes coefficient penalties comparable across feature scales.

In [1]:
import time
import warnings
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score

SEED = 42
rng = np.random.default_rng(SEED)
X, y, true_coef = make_regression(
    n_samples=700, n_features=24, n_informative=8,
    noise=12.0, coef=True, random_state=SEED
)
# Add correlated columns to make the regularization trade-off visible.
X[:, 20] = X[:, 0] * 0.9 + rng.normal(0, 0.15, len(X))
X[:, 21] = X[:, 0] * 0.9 + rng.normal(0, 0.15, len(X))
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("Train:", X_train.shape, "Test:", X_test.shape)


Train: (525, 24) Test: (175, 24)


## 5. From-scratch implementation

The implementation centers the training data to estimate an unpenalized intercept, then fits coefficients using the objective defined above. It validates shapes and parameter ranges.

In [2]:
def predict_linear(X, intercept, coef):
    return np.asarray(X) @ coef + intercept

def soft_threshold(z, threshold):
    return np.sign(z) * np.maximum(np.abs(z) - threshold, 0.0)

def fit_lasso_cd(X, y, alpha=0.05, max_iter=20000, tol=1e-8):
    X = np.asarray(X, dtype=float); y = np.asarray(y, dtype=float)
    if X.ndim != 2 or y.ndim != 1 or len(X) != len(y) or len(X) == 0:
        raise ValueError("X must be non-empty 2D and y must be matching 1D")
    if alpha < 0 or max_iter < 1 or tol < 0:
        raise ValueError("Invalid alpha, max_iter, or tol")
    xm, ym = X.mean(axis=0), y.mean()
    Xc, yc = X - xm, y - ym
    n, d = Xc.shape
    w = np.zeros(d)
    norms = np.sum(Xc * Xc, axis=0)
    for iteration in range(max_iter):
        max_change = 0.0
        for j in range(d):
            if norms[j] == 0:
                w[j] = 0.0
                continue
            residual_without_j = yc - Xc @ w + Xc[:, j] * w[j]
            new_wj = soft_threshold(Xc[:, j] @ residual_without_j, n * alpha) / norms[j]
            max_change = max(max_change, abs(new_wj - w[j]))
            w[j] = new_wj
        if max_change < tol:
            break
    b = ym - xm @ w
    return b, w, iteration + 1

alpha = 0.05
b_scratch, w_scratch, n_iter = fit_lasso_cd(X_train, y_train, alpha=alpha)
pred_scratch = predict_linear(X_test, b_scratch, w_scratch)
print("Iterations:", n_iter, "nonzero coefficients:", int(np.count_nonzero(np.abs(w_scratch) > 1e-8)))
print("From-scratch MSE:", mean_squared_error(y_test, pred_scratch))
print("From-scratch R²:", r2_score(y_test, pred_scratch))


Iterations: 269 nonzero coefficients: 22
From-scratch MSE: 167.3821105410749
From-scratch R²: 0.9887019038085709


## 6. Compare with scikit-learn

Compare predictions, not just coefficient vectors. Coefficients can be affected by solver tolerance and collinearity. The library and scratch implementations below use matching objective conventions.

In [3]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    model = Lasso(alpha=alpha, max_iter=50000, tol=1e-8, selection="cyclic").fit(X_train, y_train)
pred_sklearn = model.predict(X_test)
print("scikit-learn MSE:", mean_squared_error(y_test, pred_sklearn))
print("scikit-learn nonzero coefficients:", int(np.count_nonzero(np.abs(model.coef_) > 1e-8)))
print("Max prediction difference:", np.max(np.abs(pred_scratch - pred_sklearn)))
assert np.allclose(pred_scratch, pred_sklearn, atol=5e-3, rtol=5e-3)


scikit-learn MSE: 167.3821102454236
scikit-learn nonzero coefficients: 22
Max prediction difference: 1.1579648720783098e-05


## 7. Hyperparameters

`alpha` controls sparsity. Increasing it usually shrinks more coefficients to zero. Tune it with cross-validation; when predictors are correlated, inspect coefficient stability and consider Elastic Net.

## 8. Best practices

Scale features; inspect convergence warnings; use cross-validation; treat selected features cautiously under correlation; keep a final untouched test set; consider Elastic Net when features form correlated groups.

## 9. Time/space complexity

A cyclic coordinate-descent sweep costs O(nd) for dense X; with q sweeps, about O(qnd). The number of sweeps depends on conditioning and tolerance. Inference is O(md), model storage O(d). Sparse-aware implementations can exploit sparse X.

## 10. Scaling to millions of samples

Use float32 when accuracy allows, sparse-aware solvers for sparse matrices, and iterative/minibatch methods for large n. Avoid materializing copies unnecessarily. For very high dimensions, choose solvers based on the relationship between n and d. Benchmark quality, runtime and peak memory together.

## 11. GPU suitability

A GPU is most promising when the implementation is dominated by large dense matrix operations. Coordinate descent updates one coefficient at a time, so it may be less GPU-friendly than proximal-gradient optimization. Always include data transfer and device memory in timing.

In [4]:
# Coordinate descent has sequential coordinate updates, so a direct GPU port may not help much.
# Proximal gradient is more GPU-parallel; below is an optional compact demonstration.
try:
    import torch
    if torch.cuda.is_available():
        dev = "cuda"
        Xt = torch.as_tensor(X_train, dtype=torch.float32, device=dev)
        yt = torch.as_tensor(y_train, dtype=torch.float32, device=dev)
        xm, ym = Xt.mean(0), yt.mean()
        Xc, yc = Xt-xm, yt-ym
        L = torch.linalg.matrix_norm(Xc.T @ Xc / len(Xc), ord=2).item()
        w = torch.zeros(Xc.shape[1], device=dev)
        lr = 1.0 / max(L, 1e-6)
        for _ in range(3000):
            grad = Xc.T @ (Xc @ w - yc) / len(Xc)
            w = soft_threshold((w - lr*grad).cpu().numpy(), lr*alpha)
            w = torch.as_tensor(w, dtype=torch.float32, device=dev)
        print("GPU proximal-gradient demo completed; benchmark accuracy/runtime before preferring it.")
    else:
        print("CUDA unavailable; GPU demonstration skipped.")
except ImportError:
    print("PyTorch is optional and is not installed.")


CUDA unavailable; GPU demonstration skipped.


## 12. Correctness tests and synthesis

The regularizer trades coefficient flexibility for stability. Ridge shrinks coefficients continuously, Lasso can make them exactly zero, and Elastic Net interpolates between these behaviors. The validation strategy matters as much as the optimizer.

In [5]:
assert np.isfinite(pred_scratch).all()
assert len(pred_scratch) == len(y_test)
assert mean_squared_error(y_test, pred_scratch) < np.var(y_test)
try:
    fit_lasso_cd(np.array([1, 2, 3]), np.array([1, 2, 3]))
except ValueError:
    pass
else:
    raise AssertionError("Expected invalid X dimensionality to raise")
print("All Lasso checks passed.")


All Lasso checks passed.
